# Urban Mobility & Economic Productivity

**Academic portfolio edition · TripleTen Data Analytics Bootcamp · 2026**

This notebook examines urban traffic congestion and GDP per capita across 15 cities in 7 countries, using 2024 data. It retains the analytical steps of the approved project.

**Data inputs:** `tomtom_traffic.csv` and `oecd_city_economy.csv`. The original source files are not included in this notebook; place appropriately licensed copies in `data/raw/` before running.

**Important:** The saved outputs are from the original approved analysis. This portfolio edition has not been re-executed because the two raw input datasets were not provided.


In [ ]:
#importar librerias
import pandas as pd
import numpy as nu
import seaborn as sb
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
traffic = pd.read_csv('data/raw/tomtom_traffic.csv')
eco = pd.read_csv('data/raw/oecd_city_economy.csv')

In [ ]:
traffic.head()

In [ ]:
eco.head()

## 1. Inspect source data

Review column types and the available traffic and economic variables.


In [ ]:
# Examinar la estructura de traffic
traffic.info()

In [ ]:
# Examinar la estructura de eco
eco.info()

## 2. Standardize and clean data

Rename columns, convert timestamps and numeric fields, and inspect the cleaned structure.


In [ ]:
# Estandarizar los nombres de las columnas de traffic
traffic = traffic.rename(columns={'City': 'city','Country': 'country', 'UpdateTimeUTC': 'update_time_utc', 'UpdateTimeUTCWeekAgo': 'update_time_utc_week_ago', 'JamsDelay': 'jams_delay', 'TrafficIndexLive': 'traffic_index_live','JamsLengthInKms': 'jams_length_kms', 'JamsCount': 'jams_count', 'MinsDelay': 'mins_delay', 'TravelTimeLivePer10KmsMins': 'travel_time_live_per_10kms_mins', 'TravelTimeHistoricPer10KmsMins': 'travel_time_hist_per_10kms_mins', 'TrafficIndexWeekAgo': 'traffic_index_week_ago'})

# verificar cambios
traffic.columns

In [ ]:
# Estandarizar los nombres de las columnas de eco
eco = eco.rename(columns={'Country': 'country', 'PM2.5 (μg/m³)': 'pm25', 'City GDP/capita': 'city_gdp_capita', 'Unemployment %': 'unemployment_pct', 'Population (M)': 'population_m'})
# verificar cambios
eco.columns

In [ ]:
# Convertir las columnas de traffic a tipo fecha con pd.to_datetime()
traffic['update_time_utc'] = pd.to_datetime( traffic['update_time_utc'], errors= 'coerce')
traffic['update_time_utc_week_ago'] = pd.to_datetime( traffic['update_time_utc_week_ago'], errors= 'coerce')

# verificar el cambio
traffic.info()

In [ ]:
# Limpia separadores y convierte columnas numéricas en eco
eco['city_gdp_capita'] = ( eco['city_gdp_capita']
.astype(str)
.str.replace('.', '')
.str.replace(',', '.')
.astype(float)
)
eco['unemployment_pct'] = (
eco['unemployment_pct']
.astype(str)
.str.replace('%','')
.str.replace(',', '.')
.astype(float)
)
eco['population_m'] = (
eco['population_m']
.astype(str)
.str.replace(',', '.')
.astype(float)
)

# Calcula la población total en unidades absolutas (Multiplica * 1000000)
eco['population'] = eco['population_m'] * 1000000

# verificar el cambio
eco.info()
eco.head(3)

## 3. Filter to 2024

Extract the traffic observation year and filter both sources to the analysis period.


In [ ]:
# Extraer el año de las fechas en update_time_utc
traffic['year'] = traffic['update_time_utc'].dt.year

# Verificar cambio
traffic.head(3)

In [ ]:
# Filtra los registros del año 2024
traffic_2024 = traffic[traffic['year'] == 2024].copy()
eco_2024 = eco[eco['Year'] == 2024].copy()

# Revisar dataframes nuevos
display(traffic_2024.head())
display(eco_2024.head())

## 4. Aggregate traffic indicators

Compute mean traffic indicators at the city–country–year level.


In [ ]:

# Calcular los  promedios de trafico por ciudad, país y año
traffic_city_year_2024 =  (traffic_2024
   .groupby(['city', 'country', 'year'])
    .agg({
     'jams_delay': 'mean',
      'traffic_index_live': 'mean',
       'jams_length_kms': 'mean',
        'jams_count': 'mean',
        'mins_delay': 'mean',
        'travel_time_live_per_10kms_mins': 'mean',
        'travel_time_hist_per_10kms_mins': 'mean'
        })
    .reset_index())
   

# Mostrar resultado
traffic_city_year_2024.head()




In [ ]:
traffic_city_year_2024.sort_values(["jams_delay"], ascending=False)

## 5. Merge traffic and economic data

Use an inner join on city and year to retain matching observations.


In [ ]:

# Seleccionar columnas clave de tráfico y economía
left_cols = ['city','country','year','jams_delay','traffic_index_live',
             'jams_length_kms','jams_count','mins_delay',
             'travel_time_live_per_10kms_mins','travel_time_hist_per_10kms_mins']

right_cols = ['City','Year','city_gdp_capita','unemployment_pct','pm25','population']

# Usar .copy() para crear los dos nuevos datasets reducidos
traffic_2024_small = traffic_city_year_2024[left_cols].copy()
eco_2024_small = eco_2024[right_cols].copy()

# Unir datasets
merged = traffic_2024_small.merge(
    eco_2024_small,
    left_on=['city', 'year'],
    right_on=['City', 'Year'],
    how='inner'
)

# Mostrar las primeras 5 filas
merged.head()


## 6. Exploratory visualizations

Inspect congestion distribution, GDP-per-capita distribution and city-level comparisons.


In [ ]:
# Crear boxplot para observar el comportamiento de los minutos de congestion JamsDelay
sb.boxplot(data=merged, y='jams_delay', showmeans=True)

# obtener promedio para mostrarlo en título
mean_value = merged['jams_delay'].mean()
plt.title(f'Boxplot de JamsDelay (2024)\nPromedio: {mean_value:.2f}')
plt.ylabel('Minutos de congestión')
plt.grid(axis='y', alpha=0.3)
plt.show()


In [ ]:
# Crear histograma para ver la distribución de la economía (city_gdp_capita)
mean_gdp = merged['city_gdp_capita'].mean()

plt.hist(merged['city_gdp_capita'], bins=10, edgecolor='white')

plt.axvline(mean_gdp, linestyle='--',
            label=f'Promedio: {mean_gdp:.2f}')

plt.title('Distribución del PIB per cápita (2024)')
plt.xlabel('PIB per cápita')
plt.ylabel('Número de ciudades')
plt.legend()

plt.show()

#Se redujo la cantidad de bins a la mitad (10) debido a que los 20 intervalos generaban espacios vacíos (son pocas ciudades) y esto provocaba que la distribución,
#se viera fragmentada.

In [ ]:
# Gráfico de barras para comparar jams_delay y city_gdp_capita por ciudad

ax = merged.plot(
    'City',
    y=['jams_delay', 'city_gdp_capita'],
    kind='bar',
    figsize=(12,6)
)

plt.title('Congestión y PIB per cápita por ciudad (2024)')
plt.xlabel('Ciudad')
plt.ylabel('Valor')
plt.xticks(rotation=90)
plt.legend(['Minutos de congestión', 'PIB per cápita'])

plt.tight_layout()
plt.show()



## 7. Export analytical dataset

Save the merged 2024 city-level dataset.


In [ ]:
# Exporta el dataset final como CSV
merged.to_csv("data/processed/ladb_mobility_economy_2024_clean.csv", index=False)

## 8. Key Findings and Recommendations

The approved analysis reported the following Pearson correlations:

| Indicators | r | Interpretation |
|---|---:|---|
| Congestion delay vs. GDP per capita | 0.28 | Weak positive association |
| Traffic index vs. GDP per capita | 0.06 | Negligible association |
| Travel time per 10 km vs. GDP per capita | 0.09 | Negligible association |
| Traffic index vs. travel time | 0.84 | Strong positive association |

The analysis did not find a strong negative linear relationship between congestion and GDP per capita in this sample. Bogotá combined above-average congestion with below-average GDP per capita, making it a candidate for further assessment. Santiago also met these exploratory criteria, with lower congestion than Bogotá.

**Limitations:** Correlation is not causation. Investment decisions require additional factors such as population, employment, infrastructure conditions and multi-year data.

**Reproducibility note:** The notebook has been reorganized for public presentation, and the data paths have been adapted to a proposed repository structure. Results have not been recalculated in this edition.
